# 02 ARIMA Modeling

Fit ARIMA model to capture mean dynamics of log returns.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.config import (
    ARIMA_MAX_P, ARIMA_MAX_D, ARIMA_MAX_Q, DATA_START_DATE, DATA_END_DATE, TICKER
)
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load data
price_data = download_data(ticker=TICKER, start=DATA_START_DATE, end=DATA_END_DATE)
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns().dropna()

print(f"Data loaded: {len(returns)} observations")

In [ ]:
# Fit ARIMA using auto_arima
logger.info(f"Fitting ARIMA(max_p={ARIMA_MAX_P}, max_d={ARIMA_MAX_D}, max_q={ARIMA_MAX_Q})")

arima_model = auto_arima(
    returns,
    max_p=ARIMA_MAX_P,
    max_d=ARIMA_MAX_D,
    max_q=ARIMA_MAX_Q,
    seasonal=False,
    stepwise=True,
    information_criterion='aic',
    trace=True
)

print(f"\nBest ARIMA order: {arima_model.order}")
print(f"AIC: {arima_model.aic():.4f}")

In [ ]:
# Summary and diagnostics
print(arima_model.summary())

# Plot diagnostics
arima_model.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.savefig('../report/figures/02_arima_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info("ARIMA diagnostics complete")

In [ ]:
# Forecast residuals for further GARCH modeling
fitted_values, residuals = arima_model.get_fitted_values(return_confidence_intervals=False), arima_model.resid()

# These residuals will be used in GARCH modeling
print(f"Residual mean: {residuals.mean():.6f}")
print(f"Residual std: {residuals.std():.6f}")
print(f"Residual skewness: {residuals.skew():.4f}")
print(f"Residuals ready for GARCH modeling")